In [1]:
import logging

import torch
from torch.utils.data import DataLoader
from diffusers.models import AutoencoderKL

from torchvision.datasets import ImageFolder
from torchvision import transforms

import numpy as np
from collections import OrderedDict
from copy import deepcopy
from time import time

from references.DiT.diffusion import create_diffusion
from references.DiT.models import DiT_models

/Users/diab/Desktop/Projects/VoT/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)

In [ ]:

TransTrak;ldk;asfrom PIL import Image

#################################################################################
#                             Training Helper Functions                         #
#################################################################################

@torch.no_grad()
def update_ema(ema_model, model, decay=0.9999):
    """
    Step the EMA model towards the current model.
    """
    ema_params = OrderedDict(ema_model.named_parameters())
    model_params = OrderedDict(model.named_parameters())

    for name, param in model_params.items():
        # TODO: Consider applying only to params that require_grad to avoid small numerical changes of pos_embed
        ema_params[name].mul_(decay).add_(param.data, alpha=1 - decay)


def requires_grad(model, flag=True):
    """
    Set requires_grad flag for all parameters in a model.
    """
    for p in model.parameters():
        p.requires_grad = flag

def create_logger():
    """
    Create a logger that writes to a log file and stdout.
    """
    logging.basicConfig(
        level=logging.INFO,
        format='[\033[34m%(asctime)s\033[0m] %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
        handlers=[logging.StreamHandler()]
    )
    logger = logging.getLogger(__name__)
    return logger

def center_crop_arr(pil_image, image_size):
    """
    Center cropping implementation from ADM.
    https://github.com/openai/guided-diffusion/blob/8fb3ad9197f16bbc40620447b2742e13458d2831/guided_diffusion/image_datasets.py#L126
    """
    while min(*pil_image.size) >= 2 * image_size:
        pil_image = pil_image.resize(
            tuple(x // 2 for x in pil_image.size), resample=Image.BOX
        )

    scale = image_size / min(*pil_image.size)
    pil_image = pil_image.resize(
        tuple(round(x * scale) for x in pil_image.size), resample=Image.BICUBIC
    )

    arr = np.array(pil_image)
    crop_y = (arr.shape[0] - image_size) // 2
    crop_x = (arr.shape[1] - image_size) // 2
    return Image.fromarray(arr[crop_y: crop_y + image_size, crop_x: crop_x + image_size])



In [4]:



logger = create_logger()

args = {
    "model": "DiT-S/2",
    "num_classes": 10,
    "image_size": 256,
    "data_path": "../../data/imagenette2-160/train", 
    "epochs": 100,
    "log_every": 50,
    "ckpt_every": 2000,
    "global_batch_size": 64,
    "num_workers": 4,
    "global_seed": 0,
}

assert args["image_size"] % 8 == 0, "Image size must be divisible by 8 (for the VAE encoder)."

latent_size = args["image_size"] // 8
model = DiT_models[args["model"]](
    input_size = latent_size,
    num_classes = args["num_classes"]
)

# Note that parameter initialization is done within the DiT constructor
ema = deepcopy(model).to(device)
requires_grad(ema, False)

model = model.to(device)
diffusion = create_diffusion(timestep_respacing="")

vae_model = "stabilityai/sd-vae-ft-ema"
vae = AutoencoderKL.from_pretrained(vae_model).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
/Users/diab/Desktop/Projects/VoT/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
[2026-03-08 16:03:54] HTTP Request: HEAD https://huggingface.co/stabilityai/sd-vae-ft-ema/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-03-08 16:03:55] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/stabilityai/sd-vae-ft-ema/f04b2c4b98319346dad8c65879f680b1997b204a/config.json "HTTP/1.1 200 OK"
[2026-03-08 16:03:55] HTTP Request: HEAD https://huggingface.co/stabilitya

In [5]:
# Setup data:

transform = transforms.Compose([
    transforms.Lambda(lambda pil_image: center_crop_arr(pil_image, args["image_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
])

dataset = ImageFolder(args["data_path"], transform=transform)

loader = DataLoader(
    dataset,
    batch_size=int(args["global_batch_size"]),
    shuffle=False,
    pin_memory=True,
    drop_last=True
)

logger.info(f"Dataset contains {len(dataset):,} images ({args["data_path"]})")

[2026-03-08 16:03:56] Dataset contains 9,469 images (../../data/imagenette2-160/train)


In [6]:
# Prepare models for training:
update_ema(ema, model, decay=0)  # Ensure EMA is initialized with synced weights
model.train()  # important! This enables embedding dropout for classifier-free guidance
ema.eval()  # EMA model should always be in eval mode

DiT(
  (x_embedder): PatchEmbed(
    (proj): Conv2d(4, 384, kernel_size=(2, 2), stride=(2, 2))
    (norm): Identity()
  )
  (t_embedder): TimestepEmbedder(
    (mlp): Sequential(
      (0): Linear(in_features=256, out_features=384, bias=True)
      (1): SiLU()
      (2): Linear(in_features=384, out_features=384, bias=True)
    )
  )
  (y_embedder): LabelEmbedder(
    (embedding_table): Embedding(11, 384)
  )
  (blocks): ModuleList(
    (0-11): 12 x DiTBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=False)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=False)
      (mlp): Mlp(
        (fc1): Linear

In [7]:
# Variables for monitoring/logging purposes:
train_steps = 0
log_steps = 0
running_loss = 0
start_time = time()

In [10]:


logger.info(f"Training for {args["epochs"]} epochs...")

for epoch in range(args["epochs"]):
    logger.info(f"Beginning epoch {epoch}...")
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        with torch.no_grad():
            # Map input images to latent space + normalize latents:
            x = vae.encode(x).latent_dist.sample().mul_(0.18215)

        t = torch.randint(0, diffusion.num_timesteps, (x.shape[0],), device=device)
        model_kwargs = dict(y=y)
        loss_dict = diffusion.training_losses(model, x, t, model_kwargs)
        loss = loss_dict["loss"].mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        update_ema(ema, model)

        # Log loss values:
        running_loss += loss.item()
        log_steps += 1
        train_steps += 1
        if train_steps % args["log_every"] == 0:
            # Measure training speed:
            end_time = time()
            steps_per_sec = log_steps / (end_time - start_time)
            # Reduce loss history over all processes:
            avg_loss = torch.tensor(running_loss / log_steps, device=device)
            avg_loss = avg_loss.item()
            logger.info(f"(step={train_steps:07d}) Train Loss: {avg_loss:.4f}, Train Steps/Sec: {steps_per_sec:.2f}")
            # Reset monitoring variables:
            running_loss = 0
            log_steps = 0
            start_time = time()


model.eval()
logger.info("Done!")


[2026-03-08 16:13:01] Training for 100 epochs...
[2026-03-08 16:13:01] Beginning epoch 0...
[2026-03-08 16:18:47] (step=0000100) Train Loss: 0.4739, Train Steps/Sec: 0.11
[2026-03-08 16:24:50] (step=0000150) Train Loss: 0.2271, Train Steps/Sec: 0.14
[2026-03-08 16:31:16] Beginning epoch 1...
[2026-03-08 16:31:43] (step=0000200) Train Loss: 0.2003, Train Steps/Sec: 0.12
[2026-03-08 16:38:36] (step=0000250) Train Loss: 0.2060, Train Steps/Sec: 0.12
[2026-03-08 16:45:01] (step=0000300) Train Loss: 0.2173, Train Steps/Sec: 0.13


KeyboardInterrupt: 